## Cell 1: setup

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT/"src" / "data.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Could not find src/data.py")
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
print(f"Project root:", ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data import FULL_PARQUET
from src.features import (filter_types, add_features, build_xy,
                          SET_A, SET_B, TARGET)

Project root: E:\Portifolio\MLs\mobile-money-fraud-triage


## Cell 2: loading the full file without the name columns

In [2]:
cols = ["step", "type", "amount", "oldbalanceOrg", "newbalanceOrig",
        "oldbalanceDest", "newbalanceDest", "isFraud"]
raw = pd.read_parquet(FULL_PARQUET, columns=cols)

model_df = filter_types(raw)
print(f"All rows: {len(raw):,}  frauds: {raw['isFraud'].sum():,}")
print(f"TRANSFER + CASH_OUT rows: {len(model_df):,}  "
      f"frauds: {model_df['isFraud'].sum():,}  "
      f"fraud rate: {model_df['isFraud'].mean():.4%}")

All rows: 6,362,620  frauds: 8,213
TRANSFER + CASH_OUT rows: 2,770,409  frauds: 8,213  fraud rate: 0.2965%


## Cell 3: confirming the type restriction is justified on the full data

In [3]:
print(raw.groupby("type")["isFraud"].sum())

type
CASH_IN        0
CASH_OUT    4116
DEBIT          0
PAYMENT        0
TRANSFER    4097
Name: isFraud, dtype: int64


## Cell 4: building both sets and compare

In [4]:
X_a, y = build_xy(raw, "A")
X_b, _ = build_xy(raw, "B")
print("Set A:", X_a.shape, "| Set B:", X_b.shape)
print("Columns only in B:", sorted(set(X_b.columns) - set(X_a.columns)))
X_a.describe().T[["mean", "min", "max"]]

Set A: (2770409, 10) | Set B: (2770409, 14)
Columns only in B: ['dest_balance_error', 'newbalanceDest', 'newbalanceOrig', 'orig_balance_error']


,mean,min,max
is_transfer,1.923575e-01,0.0,1.000000e+00
amount,3.175361e+05,0.0,9.244552e+07
log_amount,1.192780e+01,0.0,1.834213e+01
hour,1.530878e+01,0.0,2.300000e+01
oldbalanceOrg,4.764308e+04,0.0,5.958504e+07
oldbalanceDest,1.703551e+06,0.0,3.560159e+08
drains_account,2.895240e-03,0.0,1.000000e+00
amount_to_balance,3.006988e+00,0.0,1.000000e+01
orig_old_zero,4.723425e-01,0.0,1.000000e+00
dest_old_zero,1.405280e-01,0.0,1.000000e+00


## Cell 5: sanity check on the full data

In [5]:
feat = add_features(model_df)
print(feat.groupby("drains_account")["isFraud"].agg(["count", "sum", "mean"]))
print(feat.groupby("orig_old_zero")["isFraud"].agg(["count", "sum", "mean"]))

                  count   sum      mean
drains_account                         
0               2762388   195  0.000071
1                  8021  8018  0.999626
                 count   sum      mean
orig_old_zero                         
0              1461827  8172  0.005590
1              1308582    41  0.000031


In [6]:
feat = add_features(model_df)
print((feat["amount_to_balance"] >= 10).mean())
print(feat.groupby(feat["amount_to_balance"] >= 10)["isFraud"].agg(["count", "sum"]))

0.20562631726939956
                     count   sum
amount_to_balance               
False              2200740  8211
True                569669     2


## Feature design decisions

### Why only TRANSFER and CASH_OUT
Across the full dataset, all 8,213 frauds sit in two transaction types:
4,116 in CASH_OUT and 4,097 in TRANSFER. CASH_IN, DEBIT and PAYMENT contain
none. Modelling the other three types would add about 3.6 million rows
that can only ever be labelled "not fraud", which dilutes the class balance
and teaches the model nothing. Restricting to the two risky types leaves
2,770,409 rows with a fraud rate of 0.30%. The consequence is that this
model only scores TRANSFER and CASH_OUT, and the other types would need
their own checks if fraud appeared in them.

### Why post-transaction balances are excluded from Set A
A triage decision (approve, hold, review) has to be made while the
transaction is still pending. The balances after the transaction
(`newbalanceOrig`, `newbalanceDest`) and the balance-error features built
from them describe the outcome, so they do not exist at decision time.
Using them would mean the model peeks at the result.
The EDA also shows they are unreliable in this dataset: the origin balance
arithmetic fails for about 91% of legitimate rows but only 0.1% of fraud
rows, which is a pattern from how the data was generated, not something a
real system would record. These columns appear only in Set B, which is
labelled forensic and used for comparison, never for deployment.

### Why `drains_account` is allowed in Set A
It is built from the amount and the sender's balance before the
transaction, both known when the request arrives. Emptying an account in
one move is also a plausible real-world warning sign. However, it is
near-decisive here: 8,018 of the 8,213 frauds drain the account, and
99.96% of rows with this flag are fraud. This suggests the simulator's
fraud pattern is simple, so a high score on this dataset says little about
real M-Pesa performance. To keep results honest, I report a one-line rule
baseline and also evaluate on the rows where `drains_account = 0` (195
frauds), where a model has to do more than the rule.

### Why `step` and `isFlaggedFraud` are excluded
`step` is the hour index within the simulated month. Because I split by
time, the test steps lie outside the training range, so a model that
learned specific step values would not generalise and could pick up
periods that only exist in the simulation. I keep `hour` (step mod 24)
instead, which captures the time of day without the position in the month.
`isFlaggedFraud` is the simulator's own rule-based flag, not a property of
the transaction itself. It is set on only 16 of 6,362,620 rows (all 16 are
fraud), so it catches about 0.2% of the 8,213 frauds and adds no usable
signal. As I recall, the Kaggle description says it marks attempts to move
more than 200,000 in a single transaction, but confirm that on the dataset
page before quoting it.

### The `amount_to_balance` cap
The ratio is amount divided by the sender's balance before the transaction,
set to 0 when the balance is 0 and capped at 10. About 20.6% of rows
(569,669) hit the cap, and only 2 of them are fraud, so the cap removes
extreme values without losing signal: fraud sits near a ratio of 1, where
the whole balance is moved. The cap is a guard for the Logistic Regression
baseline, and tree models are unaffected. These rows are transactions much
larger than the sender's balance, which a real system would likely reject,
a

In [7]:
flag = pd.read_parquet(FULL_PARQUET, columns=["isFlaggedFraud", "isFraud"])
print(flag.groupby("isFlaggedFraud")["isFraud"].agg(["count", "sum"]))

                  count   sum
isFlaggedFraud               
0               6362604  8197
1                    16    16
